In [1]:
# ==========================================================
# 1. Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================================================
# 2. Load Dataset
# ==========================================================

df = pd.read_csv("dash-stock-ticker-demo.csv")

# View dataset
df.head()

,Unnamed: 0,Date,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume,Stock
0,0,2017-12-29,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0,AAPL
1,1,2017-12-28,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0,AAPL
2,2,2017-12-27,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0,AAPL
3,3,2017-12-26,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0,AAPL
4,4,2017-12-22,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0,AAPL


In [2]:
df.info()

print(df.columns)

df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3634 entries, 0 to 3633
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  3634 non-null   int64  
 1   Date        3634 non-null   str    
 2   Open        3634 non-null   float64
 3   High        3634 non-null   float64
 4   Low         3634 non-null   float64
 5   Close       3634 non-null   float64
 6   Volume      3634 non-null   float64
 7   ExDividend  3634 non-null   float64
 8   SplitRatio  3634 non-null   float64
 9   AdjOpen     3634 non-null   float64
 10  AdjHigh     3634 non-null   float64
 11  AdjLow      3634 non-null   float64
 12  AdjClose    3634 non-null   float64
 13  AdjVolume   3634 non-null   float64
 14  Stock       3634 non-null   str    
dtypes: float64(12), int64(1), str(2)
memory usage: 476.4 KB
Index(['Unnamed: 0', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
       'ExDividend', 'SplitRatio', 'AdjOpen', 'AdjHigh', 'AdjLow', 'AdjClos

,Unnamed: 0,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume
count,3634.000000,3634.000000,3634.000000,3634.000000,3634.000000,3.634000e+03,3634.000000,3634.0,3634.000000,3634.000000,3634.000000,3634.000000,3.634000e+03
mean,364.898734,280.507209,283.223866,277.610512,280.512406,1.198852e+07,0.002501,1.0,279.713117,282.421203,276.825323,279.718199,1.198852e+07
std,213.009906,271.462181,273.211793,269.443888,271.412545,1.824834e+07,0.033689,0.0,271.905755,273.660840,269.881757,271.856209,1.824834e+07
min,0.000000,26.460000,26.970000,26.150000,26.760000,6.336000e+03,0.000000,1.0,26.460000,26.970000,26.150000,26.760000,6.336000e+03
25%,181.000000,111.075000,112.192500,109.705000,111.147500,1.281084e+06,0.000000,1.0,107.743055,108.947576,106.611227,107.963149,1.281084e+06
50%,363.000000,174.240000,176.182500,172.560000,174.450000,4.113626e+06,0.000000,1.0,173.643997,175.657869,171.774585,173.863440,4.113626e+06
75%,544.750000,309.717500,313.902500,305.475000,308.880000,1.573505e+07,0.000000,1.0,309.717500,313.902500,305.475000,308.880000,1.573505e+07
max,753.000000,1083.020000,1086.490000,1072.270000,1085.090000,2.510321e+08,0.630000,1.0,1083.020000,1086.490000,1072.270000,1085.090000,2.510321e+08


In [3]:
# Remove missing values

df = df.dropna()

# Convert dates if available

for col in df.columns:
    if "date" in col.lower():
        df[col] = pd.to_datetime(df[col])

df.head()

,Unnamed: 0,Date,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume,Stock
0,0,2017-12-29,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0,AAPL
1,1,2017-12-28,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0,AAPL
2,2,2017-12-27,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0,AAPL
3,3,2017-12-26,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0,AAPL
4,4,2017-12-22,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0,AAPL


In [4]:
text_columns = []

for col in df.columns:
    if df[col].dtype == "object":
        text_columns.append(col)

text_columns

[]

In [6]:
df.head()

,Unnamed: 0,Date,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume,Stock
0,0,2017-12-29,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0,AAPL
1,1,2017-12-28,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0,AAPL
2,2,2017-12-27,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0,AAPL
3,3,2017-12-26,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0,AAPL
4,4,2017-12-22,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0,AAPL


In [7]:
numeric_features = df.select_dtypes(
    include=["int64","float64"]
)

numeric_features.head()

,Unnamed: 0,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume
0,0,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0
1,1,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0
2,2,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0
3,3,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0
4,4,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0


In [8]:
scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(
    numeric_features
)

scaled_numeric = pd.DataFrame(
    scaled_numeric,
    columns=numeric_features.columns
)

In [10]:
df.head()

,Unnamed: 0,Date,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume,Stock
0,0,2017-12-29,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0,AAPL
1,1,2017-12-28,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0,AAPL
2,2,2017-12-27,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0,AAPL
3,3,2017-12-26,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0,AAPL
4,4,2017-12-22,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0,AAPL


In [12]:
df.columns

Index(['Unnamed: 0', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume',
       'ExDividend', 'SplitRatio', 'AdjOpen', 'AdjHigh', 'AdjLow', 'AdjClose',
       'AdjVolume', 'Stock'],
      dtype='str')

In [14]:
df.columns.tolist()
df.head()

,Unnamed: 0,Date,Open,High,Low,Close,Volume,ExDividend,SplitRatio,AdjOpen,AdjHigh,AdjLow,AdjClose,AdjVolume,Stock
0,0,2017-12-29,170.52,170.590,169.220,169.23,25643711.0,0.0,1.0,170.52,170.590,169.220,169.23,25643711.0,AAPL
1,1,2017-12-28,171.00,171.850,170.480,171.08,15997739.0,0.0,1.0,171.00,171.850,170.480,171.08,15997739.0,AAPL
2,2,2017-12-27,170.10,170.780,169.710,170.60,21672062.0,0.0,1.0,170.10,170.780,169.710,170.60,21672062.0,AAPL
3,3,2017-12-26,170.80,171.470,169.679,170.57,32968167.0,0.0,1.0,170.80,171.470,169.679,170.57,32968167.0,AAPL
4,4,2017-12-22,174.68,175.424,174.500,175.01,16052615.0,0.0,1.0,174.68,175.424,174.500,175.01,16052615.0,AAPL
